In [1]:
import nest
from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils

neuron_nestml = """
model lif_STC_neuron:
    parameters:
        dt ms = 0.1 ms                                  # Simulation timestep
        tau_mem ms = 10 ms                              # Membrane time constant
        R MOhm = 10 MOhm                                # Membrane resistance
        V_rev mV = -65 mV                               # Leak reversal potential (aka resting potential)
        I0 pA = 930 pA                                  # Mean external background current that produces resting-state firing rate of ~ 5-10Hz
        refr_T ms = 2 ms                                # Duration of refractory period
        V_th mV = -55 mV                                # Spike threshold potential
        V_reset mV = -70 mV                             # Reset potential
        tau_syn ms = 5 ms                               # Synapse time constant
        sigma_wn pA*s**0.5 = 50 pA*s**0.5               # STD for Gaussian noise in I_bg and Ca-dependent fluctuations
        w_stim nC = 0.420075 nC                         # Coupling strength of synapses from putative input neurons
        N_stim integer = 0                              # Number of putative input neurons for stimulation
        f_stim Hz = 0 Hz                                # Frequency of learning/recall stimulation from putative neurons
        
        # Protein dynamics
        alpha real = 1.0                                # Protein synthesis rate
        tau_p s = 3600 s                                # Protein time constant
        p_check integer = 0                             # Set externally
        TT_aux s = 1 s

    state:
        V_m mV = V_rev
        I_bg pA = I0
        I_stim_prot pA = 0.0 pA
        refr_t ms = 0.0 ms
        p real = 0.0
        gauss_noise real = 0.0
        I_stim_total pA = 0.0 pA

    input:
        exc_spikes <- excitatory spike
        inh_spikes <- inhibitory spike
        I_stim pA <- continuous

    output:
        spike

    equations:
        kernel exc_spike_kernel = exp(-t/tau_syn)
        kernel inh_spike_kernel = exp(-t/tau_syn)

        inline I_syn_exc pA = convolve(exc_spike_kernel, exc_spikes) * pA
        inline I_syn_inh pA = convolve(inh_spike_kernel, inh_spikes) * pA

        V_m' = (V_rev - V_m + R * (I_syn_exc + I_syn_inh + I_bg + I_stim_total)) / tau_mem
        I_stim_prot' = (w_stim * N_stim * f_stim - I_stim_prot + w_stim * (N_stim * f_stim)**(0.5) * gauss_noise * (1/TT_aux)**0.5) / tau_syn
        I_bg' = (I0 - I_bg + sigma_wn * gauss_noise * (1/TT_aux)**0.5) / tau_syn
        p' = (- p + alpha * p_check) / tau_p

    update:
        I_stim_total = I_stim * I_stim_prot / pA
        refr_t -= resolution()
        gauss_noise = random_normal(0, 1/dt * ms)

        if refr_t > 0 ms:
            # neuron is absolute refractory, do not evolve V_m
            integrate_odes(I_bg, I_stim_prot)
        else:
            # neuron not refractory
            integrate_odes()

    onCondition(refr_t <= 0 ms and V_m >= V_th):
        refr_t = refr_T
        V_m = V_reset
        emit_spike()
"""

synapse_nestml = """
model STC_synapse:
    parameters:
        dt ms = 0.1 ms                                      # Simulation timestep
        h0 nC = 0.420075 nC                                 # Initial E->E coupling strength
        t_c_delay ms = 18.8 ms                              # Postsyn calcium delay to presyn spike
        c_pre real = 1.0                                    # Presyn calcium contribution
        c_post real = 0.2758                                # Postsyn calcium contribution
        tau_c ms = 48.8 ms                                  # Calcium time constant
        tau_h s = 688.4 s                                   # Early-phase time constant
        tau_z s = 3600 s                                    # Late-phase time constant
        tau_p s = 3600 s                                    # Protein time constant
        gamma_p real = 1645.6                               # Potentiation rate
        gamma_d real = 313.1                                # Depression rate
        theta_p real = 3.0                                  # Calcium threshold for potentiation
        theta_d real = 1.2                                  # Calcium threshold for depression
        sigmal_pl nC*s**0.5 = 0.290435 nC*s**0.5            # STD for plasticity fluctuations
        theta_tag nC = 0.0840149 nC                         # Tagging threshold
        syn_delay ms = 1 ms                                 # Synaptic delay
        alpha real = 1.0                                    # Protein synthesis rate
        gauss_noise real = 0.0                              
        TT_aux s = 1 s                                      # Keep unit consistency
        size_ca_buffer integer = 200                        # Default - Set externally
        
    internals:
        num_delay_steps integer = ceil(t_c_delay/dt)

    input:
        pre_spikes <- spike
        post_spikes <- spike
        mod_spikes <- spike
        p_neu real <- continuous

    output:
        spike(weight nC, delay ms)

    state:
        w nC = 0.0 nC
        h nC = h0
        c real = 0.0
        z real = 0.0
        delta_z real = 0.0
        ca_buffer[size_ca_buffer] real = 0.0
        early_ltp_check integer = 0
        early_ltd_check integer = 0
        late_ltp_check integer = 0
        late_ltd_check integer = 0
        current_idx integer = 0
        delay_idx integer = 0

    equations:
        inline ca_noise nC = (tau_h * (early_ltp_check + early_ltd_check))**0.5 * sigmal_pl * gauss_noise / TT_aux
        inline early_ltp nC = gamma_p * (1.0 nC - h) * early_ltp_check
        inline early_ltd nC = gamma_d * h * early_ltd_check
        inline late_ltp real = alpha * (1 - z) * late_ltp_check
        inline late_ltd real = alpha * (z + 0.5) * late_ltd_check
        inline h_diff nC = h - h0

        h' = (0.1 * (h0 - h) + early_ltp - early_ltd + ca_noise) / tau_h
        c' = - c / tau_c

    onReceive(pre_spikes):
        # When spike arrives, Ca release is scheduled in the future
        ca_buffer[delay_idx] += c_pre
        delta_z = (p_neu * (1 - z) * late_ltp_check - p_neu * (z + 0.5) * late_ltd_check) * (dt / tau_z)
        z = z + delta_z
        emit_spike(w, syn_delay)

    onReceive(post_spikes):
        c += c_post
        delta_z = (p_neu * (1 - z) * late_ltp_check - p_neu * (z + 0.5) * late_ltd_check) * (dt / tau_z)
        z = z + delta_z

    update:
        c += ca_buffer[current_idx]

        early_ltp_check = c >= theta_p ? 1 : 0
        early_ltd_check = c >= theta_d ? 1 : 0
        late_ltp_check = h_diff >= theta_tag ? 1 : 0
        late_ltd_check = -h_diff >= theta_tag ? 1 : 0

        w = h + h0 * z

        integrate_odes()

        ca_buffer[current_idx] = 0.0
        current_idx = (current_idx + 1) % size_ca_buffer
        delay_idx = (current_idx + num_delay_steps) % size_ca_buffer
"""

codegen_opts = {"neuron_synapse_pairs": [{"neuron": "lif_STC_neuron",
                                          "synapse": "STC_synapse",
                                          "post_ports": ["post_spikes", ["p_neu", "p"]],
                                          "vt_ports": ["mod_spikes"]}],
                "delay_variable": {"STC_synapse": "syn_delay"},
                "weight_variable": {"STC_synapse": "w"}}

module_name, neuron_model, synapse_model = \
    NESTCodeGeneratorUtils.generate_code_for(neuron_nestml,
                                             synapse_nestml,
                                             codegen_opts=codegen_opts,
                                             logging_level="DEBUG")


# Second option following neuromodulated synapse tutorial - no neuron_synape_pairs

# module_name, neuron_model, synapse_model = \
#     NESTCodeGeneratorUtils.generate_code_for(neuron_nestml,
#                                              synapse_nestml,
#                                              post_ports=["post_spikes", ["p_neu", "p"]],
#                                              mod_ports=["mod_spikes"],
#                                              logging_level="DEBUG",
#                                              codegen_opts={"delay_variable": {"STC_synapse": "syn_delay"},
#                                                            "weight_variable": {"STC_synapse": "w"}})


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.8.0-post0.dev0
 Built: Sep 26 2025 13:28:49

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



[1,GLOBAL, INFO]: List of files that will be processed:
[2,GLOBAL, INFO]: /home/pooja/nestml/master/STC_model/lif_STC_neuron.nestml
[3,GLOBAL, INFO]: /home/pooja/nestml/master/STC_model/STC_synapse.nestml
[4,GLOBAL, INFO]: Target platform code will be generated in directory: '/home/pooja/nestml/master/STC_model/target'
[5,GLOBAL, INFO]: Target platform code will be installed in directory: '/tmp/nestml_target_aj7szap5'

              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.8.0-post0.dev0
 Built: Sep 26 2025 13:28:49

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.

[6,GLOBAL, INFO]: The NEST Simulator version was automatically detected as: master
[7,GLOBAL, INFO]: Given template root path is not an absolute path. Creating the absolute path with default templates directory '/home/pooja/env/nestml_de

INFO:root:Analysing input:
INFO:root:{
    "dynamics": [
        {
            "expression": "V_m' = (V_rev - V_m + R * ((exc_spike_kernel__X__exc_spikes * 1.0) + (inh_spike_kernel__X__inh_spikes * 1.0) + I_bg + I_stim_total)) / tau_mem",
            "initial_values": {
                "V_m": "V_rev"
            }
        },
        {
            "expression": "I_stim_prot' = (w_stim * N_stim * f_stim - I_stim_prot + w_stim * (N_stim * f_stim)**(0.5) * gauss_noise * (1 / TT_aux)**0.5) / tau_syn",
            "initial_values": {
                "I_stim_prot": "0.0"
            }
        },
        {
            "expression": "I_bg' = (I0 - I_bg + sigma_wn * gauss_noise * (1 / TT_aux)**0.5) / tau_syn",
            "initial_values": {
                "I_bg": "I0"
            }
        },
        {
            "expression": "p' = ((-p) + alpha * p_check) / tau_p",
            "initial_values": {
                "p": "0.0"
            }
        },
        {
            "expression": "inh_sp

[32,GLOBAL, INFO]: Analysing/transforming model 'lif_STC_neuron_nestml__with_STC_synapse_nestml'
[33,lif_STC_neuron_nestml__with_STC_synapse_nestml, INFO, [2:0;68:0]]: Starts processing of the model 'lif_STC_neuron_nestml__with_STC_synapse_nestml'


DEBUG:root:Splitting expression alpha*p_check/tau_p - p/tau_p (symbols Matrix([[V_m], [I_stim_prot], [I_bg], [p], [inh_spike_kernel__X__inh_spikes], [exc_spike_kernel__X__exc_spikes]]))
DEBUG:root:	linear factors: Matrix([[0], [0], [0], [-1/tau_p], [0], [0]])
DEBUG:root:	inhomogeneous term: alpha*p_check/tau_p
DEBUG:root:	nonlinear term: 0.0
DEBUG:root:Shape inh_spike_kernel__X__inh_spikes: reconstituting expression -inh_spike_kernel__X__inh_spikes/tau_syn
DEBUG:root:Splitting expression -inh_spike_kernel__X__inh_spikes/tau_syn (symbols Matrix([[V_m], [I_stim_prot], [I_bg], [p], [inh_spike_kernel__X__inh_spikes], [exc_spike_kernel__X__exc_spikes]]))
DEBUG:root:	linear factors: Matrix([[0], [0], [0], [0], [-1/tau_syn], [0]])
DEBUG:root:	inhomogeneous term: 0.0
DEBUG:root:	nonlinear term: 0.0
DEBUG:root:Shape exc_spike_kernel__X__exc_spikes: reconstituting expression -exc_spike_kernel__X__exc_spikes/tau_syn
DEBUG:root:Splitting expression -exc_spike_kernel__X__exc_spikes/tau_syn (symbols

[34,GLOBAL, INFO]: Analysing/transforming synapse STC_synapse_nestml__with_lif_STC_neuron_nestml.
[35,STC_synapse_nestml__with_lif_STC_neuron_nestml, INFO, [2:0;88:0]]: Starts processing of the model 'STC_synapse_nestml__with_lif_STC_neuron_nestml'


INFO:root:Analysing input:
INFO:root:{
    "dynamics": [
        {
            "expression": "h' = (0.1 * (h0 - h) + (gamma_p * (1.0 - h) * early_ltp_check) - (gamma_d * h * early_ltd_check) + ((tau_h * (early_ltp_check + early_ltd_check))**0.5 * sigmal_pl * gauss_noise / TT_aux)) / tau_h",
            "initial_values": {
                "h": "h0"
            }
        },
        {
            "expression": "c' = (-c) / tau_c",
            "initial_values": {
                "c": "0.0"
            }
        }
    ],
    "options": {
        "output_timestep_symbol": "__h",
        "simplify_expression": "sympy.logcombine(sympy.powsimp(sympy.expand(expr)))"
    },
    "parameters": {
        "TT_aux": "1",
        "alpha": "1.0",
        "c_post": "0.2758",
        "c_pre": "1.0",
        "dt": "0.1",
        "gamma_d": "313.1",
        "gamma_p": "1645.6",
        "gauss_noise": "0.0",
        "h0": "0.420075",
        "sigmal_pl": "0.290435 * 1000.0",
        "size_ca_buffer": "200",


[36,GLOBAL, INFO]: Rendering template /home/pooja/nestml/master/STC_model/target/lif_STC_neuron_nestml.cpp
[37,GLOBAL, INFO]: Rendering template /home/pooja/nestml/master/STC_model/target/lif_STC_neuron_nestml.h
[38,lif_STC_neuron_nestml, INFO, [2:0;68:0]]: Successfully generated code for the model: 'lif_STC_neuron_nestml' in: '/home/pooja/nestml/master/STC_model/target' !
[39,GLOBAL, INFO]: Rendering template /home/pooja/nestml/master/STC_model/target/lif_STC_neuron_nestml__with_STC_synapse_nestml.cpp
[40,GLOBAL, INFO]: Rendering template /home/pooja/nestml/master/STC_model/target/lif_STC_neuron_nestml__with_STC_synapse_nestml.h
[41,lif_STC_neuron_nestml__with_STC_synapse_nestml, INFO, [2:0;68:0]]: Successfully generated code for the model: 'lif_STC_neuron_nestml__with_STC_synapse_nestml' in: '/home/pooja/nestml/master/STC_model/target' !
[42,GLOBAL, INFO]: Rendering template /home/pooja/nestml/master/STC_model/target/STC_synapse_nestml__with_lif_STC_neuron_nestml.h
[43,STC_synapse_ne

GeneratedCodeBuildException: Error occurred during 'make all'! More detailed error messages can be found in stderr.